# Trabajo Ev 1

## Leer DataSet


In [1]:
import pandas as pd
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

raw_df = pd.read_csv("retail_store_sales.csv")
#respaldo y copia del dataset original

df = raw_df
df.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


##Limpieza de | Price Per Unit | Quantity | Total Spent (Busqueda de Nulos)

In [2]:


#comprobamos la cantidad de campos nulos
df.isna().sum()


,0
Transaction ID,0
Customer ID,0
Category,0
Item,1213
Price Per Unit,609
Quantity,604
Total Spent,604
Payment Method,0
Location,0
Transaction Date,0


Podemos concluir que  TODOS los registros tienen un "TRANSACTION_ID", "CUSTOMER_ID", "CATEGORY","PAYMENT METHOD",
"LOCATION" y "TRANSACTION DATE".

In [3]:
#comprobamos si existen transacciones repetidas
id_repetidos= raw_df['Transaction ID'].duplicated().sum()
#comprobamos si en general existen  registros repetidos (aunque si no hay transacciones repetidas esto se puede evitar)
datos_Repetidos = raw_df.duplicated().sum()

print(f"la cantidad de transacciones repetidas es: {id_repetidos}")
print(f"la cantidad de datos repetidos es: {datos_Repetidos}")

la cantidad de transacciones repetidas es: 0
la cantidad de datos repetidos es: 0


no existen errores de duplicación y cada transacción es unica

In [4]:
# En los siguientes 4 codigos se buscarán patrones en los que no se pueden recuperar los valores de las columnas "Price Per Unit"	"Quantity"	"Total Spent"
# Esto debido a que no por ejemplo , no podemos sacar el total sin la cantidad ni el precio unitario
#en general si hay menos de 2 campos de estas 3 columnas, el dato es irrecuperable.
# Busqueda de si "Precio", "Cantidad", "Total" son Nulos de manera simultanea

#Casos donde 'Price per unit', 'Quantity' y 'Total spent' son nulos
filtro_null_1 = df.loc[
      df['Price Per Unit'].isnull() & df['Quantity'].isnull() & df['Total Spent'].isnull(),
      ['Price Per Unit', 'Quantity', 'Total Spent']
    ]
filtro_null_1.shape

(0, 3)

In [5]:
#Casos donde 'Cantidad' y 'Total' son Nulos
filtro_null_2 = df.loc[
      df['Quantity'].isnull() & df['Total Spent'].isnull(),
      ['Price Per Unit', 'Quantity', 'Total Spent']
    ]
filtro_null_2.shape

(604, 3)

In [6]:
#Casos donde 'Price per Unit y 'Quantity' son Nulos
filtro_null_3 = df.loc[
      df['Price Per Unit'].isnull() & df['Quantity'].isnull(),
      ['Price Per Unit', 'Quantity', 'Total Spent']
    ]
filtro_null_3.shape

(0, 3)

In [7]:

#Casos donde 'Price per Unit y 'Total' son Nulos
filtro_null_4 =df.loc[
      df['Price Per Unit'].isnull() & df['Total Spent'].isnull(),
      ['Price Per Unit', 'Quantity', 'Total Spent']
    ]
filtro_null_4.shape

(0, 3)

Podemos observar que de los 4 escenarios posibles donde NO se pueden recuperar estos 3 datos, solo hay  604 escenarios donde no hay cantidad, ni precio unitario, en el resto de casos es recuperable la información por ende, solo limpiamos ese escenario.

In [8]:
#borramos los escenarios de 'Price per unit', 'Quantity' y 'Total spent' donde no es recuperable
df.drop(filtro_null_2.index, axis=0, inplace=True)
#revisamos los nulls denuevo
df.isna().sum()

,0
Transaction ID,0
Customer ID,0
Category,0
Item,609
Price Per Unit,609
Quantity,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,0


##Recuperando campos vacios

Podemos observar del bloque de codigo anterior que los unicos campos nulos actuales entre 'Price Per Unit', 'Quantity' y 'Total Spent' solo hay 609 casos nulos de Price Per unit


In [9]:

#Buscaremos reparar los campos vacios de 'Price Per unit' donde NO hay descuento (ya que el descuento nos puede dar un valor sesgado del total real)

#PRICE PER UNIT NULL

df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Total Spent'] / df['Quantity'])

df.isna().sum()



,0
Transaction ID,0
Customer ID,0
Category,0
Item,609
Price Per Unit,0
Quantity,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,0


AHORA intentaremos recuperar los espacios nulos de items




In [10]:
#Contamos la cantidad de categorias que existen
cat_size = df['Category'].value_counts()
print(cat_size)

Category
Furniture                             1525
Electric household essentials         1516
Milk Products                         1513
Food                                  1507
Butchers                              1496
Beverages                             1496
Computers and electric accessories    1477
Patisserie                            1441
Name: count, dtype: int64


In [11]:
#items unicos por cada categoria
cat_unique = df.groupby('Category')['Item'].nunique()
print(cat_unique)

Category
Beverages                             25
Butchers                              25
Computers and electric accessories    25
Electric household essentials         25
Food                                  25
Furniture                             25
Milk Products                         25
Patisserie                            25
Name: Item, dtype: int64


In [12]:
#items unicos
items_unique = df['Item'].unique()
print(len(items_unique))

201


In [13]:
# Esto te dirá qué ítems están en más de una categoría
duplicates = df.groupby('Item')['Category'].nunique()
items_en_varias_cats = duplicates[duplicates > 1]

print(f"Hay {len(items_en_varias_cats)} ítems que aparecen en más de una categoría.")

Hay 0 ítems que aparecen en más de una categoría.


## Transformacion Manual

In [ ]:
# Solo para pruebas , Eliminacion de nulos para probar transformacion manual
# RECALCANDO , ESTE CODIGO ES SOLO Y UNICAMENTE PARA BORRAR TODOS LOS NULOS SOLO CON EL FIN DE
# PROBAR LA TRANSFORMACION MANUAL PARA ADELANTAR
df_manual_prueba = df.copy()

filtro_prueba_manual = df.loc[
      df["Transaction ID"].isnull() |	df["Customer ID"].isnull() |	df["Category"].isnull() |	df["Item"].isnull() |
      df["Price Per Unit"].isnull() |	df["Quantity"].isnull() |	df["Total Spent"].isnull() | df["Payment Method"].isnull() |
      df["Location"].isnull() |	df["Transaction Date"].isnull() |	df["Discount Applied"].isnull(),
      ["Transaction ID"]
    ]
filtro_prueba_manual.shape

df_manual_prueba.drop(filtro_prueba_manual.index, axis=0, inplace=True)
df_manual_prueba.isna().sum()
df_manual_prueba.shape
df_manual_prueba.head(5)

In [37]:
# Imports para la transformacion manual y automatica para IA
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [38]:
df_manual = df_manual_prueba.copy()

# Label Encoding (Palabras a Números secuenciales)
le = LabelEncoder()
df_manual['Descuento_Aplicado'] = le.fit_transform(df_manual['Discount Applied'])

# Min-Max Scaling (Escalar entre 0 y 1)
mm_scaler = MinMaxScaler()
df_manual['Precio_Unitario_Norm'] = mm_scaler.fit_transform(df_manual[['Price Per Unit']])
df_manual['Cantidad_Norm'] = mm_scaler.fit_transform(df_manual[['Quantity']])
df_manual['Total_Norm'] = mm_scaler.fit_transform(df_manual[['Total Spent']])

# Solo consideramos las 4 columnas usadas , ya que las columnas como Customer o Category entre otros
# Tienen tantos datos que generaria muchas columnas asia la derecha , de esta forma la informacion
# Se ve mas limpia en la tabla

df_manual.head(5)


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied,Descuento_Aplicado,Precio_Unitario_Norm,Cantidad_Norm,Total_Norm
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True,1,0.375000,1.000000,0.444444
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True,1,0.666667,0.888889,0.632099
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False,0,0.458333,0.111111,0.093827
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False,0,0.208333,0.666667,0.203704
6,TXN_3652209,CUST_07,Food,Item_1_FOOD,5.0,8.0,40.0,Credit Card,In-store,2023-06-10,True,1,0.000000,0.777778,0.086420


## Traspasando para IA

In [39]:
# Codificar descuento y escalar precio unitario, cantidad y total, se usan estas columnas por la razon
# Explicada en la parte manual , si bien , no es una tabla esta vez, la idea es que sea agradable de forma visible
# Para luego poder presentarlo , Tambien se entrega como tipo "Lista" , porque asi
# Los datos estan listos para ser procesados

df_auto = df_manual_prueba.copy()

pipeline = ColumnTransformer(transformers=[
('cat', OneHotEncoder(), ['Discount Applied']),
('num', StandardScaler(), ["Price Per Unit",	"Quantity",	"Total Spent"])
])
X_transf = pipeline.fit_transform(df_auto)
print(X_transf)

[[ 0.          1.         -0.46049625  1.56247296  0.5782632 ]
 [ 0.          1.          0.52049907  1.21077617  1.38358309]
 [ 1.          0.         -0.18021187 -1.25110135 -0.92641343]
 ...
 [ 1.          0.         -1.72177595  1.21077617 -0.9052208 ]
 [ 1.          0.         -1.58163376  1.21077617 -0.76217056]
 [ 0.          1.         -0.88092282  0.15568581 -0.49196455]]
